# Creating an MCP Client 

In the [previous notebook](https://github.com/Alireza-Akhavan/Agentic_AI/blob/main/04_02_Creating_an_MCP_Server.ipynb), you created an MCP research server that exposes 2 tools. In this notebook, you will make the chatbot communicate to the server through an MCP client. This will make the chatbot MCP compatible. You will continue from where you left off in previos part, i.e., you are provided again with the `mcp_project` folder that contains the `research_server.py` file. You'll add to it the MCP chatbot file and update the environment. 

## Back to the Chatbot Example

Here are the main code parts (`process_query`, `chat_loop`) from the chatbot example in [Notebook 4_01](https://github.com/Alireza-Akhavan/Agentic_AI/blob/main/04_01_Chatbot_Example_no_MCP.ipynb). Notice that the burden of tool definitions and execution is now shifted onto the MCP server, so the chatbot logic only contains code related to processing the user queries and to keeping the chat loop running until the user types `quit`.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")


client = OpenAI(
    api_key=openai_api_key,   
    base_url=openai_base_url)

def process_query(query):
    messages = [{'role': 'user', 'content': query}]
    response = client.chat.completions.create(
        model='gpt-5-nano',
        tools=tools,
        messages=messages
    )
    process_flag = True
    while process_flag:
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if finish_reason == 'stop' or not message.tool_calls:
            print(message.content)
            process_flag = False

        elif finish_reason == 'tool_calls':
            messages.append(message)
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_call_id = tool_call.id

                print(f"Calling tool {tool_name} with args {tool_args}")
                result = execute_tool(tool_name, tool_args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": result
                })

            response = client.chat.completions.create(
                model='gpt-5-nano',
                tools=tools,
                messages=messages
            )
            if response.choices[0].finish_reason == 'stop':
                print(response.choices[0].message.content)
                process_flag = False


def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")


: 

## Building your MCP Client

Now you will take the functions `process_query` and `chat_loop` and wrap them in a `MCP_ChatBot` class. To enable the chatbot to communicate to the server, you will add a method that connects to the server through an MCP client, which follows the structure given in this reference code:

### Reference Code
``` python
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

# Create server parameters for stdio connection
server_params = StdioServerParameters(
    command="python",  # Executable
    args=["example_server.py"],  # Command line arguments
    env=None,  # Optional environment variables
)

async def run():
    # Launch the server as a subprocess & returns the read and write streams
    # read: the stream that the client will use to read msgs from the server
    # write: the stream that client will use to write msgs to the server
    async with stdio_client(server_params) as (read, write): 
        # the client session is used to initiate the connection 
        # and send requests to server 
        async with ClientSession(read, write) as session:
            # Initialize the connection (1:1 connection with the server)
            await session.initialize()

            # List available tools
            tools = await session.list_tools()

            # will call the chat_loop here
            # ....
            
            # Call a tool: this will be in the process_query method
            result = await session.call_tool("tool-name", arguments={"arg1": "value"})


if __name__ == "__main__":
    asyncio.run(run())
`````

### Adding MCP Client to the Chatbot

The MCP_ChatBot class consists of the methods:
- `process_query`
- `chat_loop`
- `connect_to_server_and_run`
  
and has the following attributes:
- session (of type ClientSession)
- openai_client: OpenAI                           
- available_tools

In `connect_to_server_and_run`, the client launches the server and requests the list of tools that the server provides (through the client session). The tool definitions are stored in the variable `available_tools` and are passed in to the LLM in `process_query`.

<img src="images/tools_discovery.png" width="400">


In `process_query`, when the LLM decides it requires a tool to be executed, the client session sends to the server the tool call request. The returned response is passed in to the LLM. 

<img src="images/tool_invocation.png" width="400">


Here're the `mcp_chatbot` code.

In [ ]:
%%writefile mcp_project/mcp_chatbot.py
from dotenv import load_dotenv
from openai import OpenAI
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from typing import List
import asyncio
import nest_asyncio
import json

nest_asyncio.apply()

load_dotenv()

class MCP_ChatBot:

    def __init__(self):
        self.session: ClientSession = None
        self.openai_client = OpenAI()
        self.available_tools: List[dict] = []

    async def process_query(self, query):
        # Build dynamic system prompt based on available tools
        tool_info = "\n".join(
            f"- {t['function']['name']}: {t['function']['description']}"
            for t in self.available_tools
        )
        system_prompt = (
            "You are a research assistant. You have access to these tools:\n"
            f"{tool_info}\n"
            "When greeted, introduce yourself and your capabilities."
        )

        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': query}
        ]

        response = self.openai_client.chat.completions.create(
            model='gpt-5-nano',
            tools=self.available_tools,
            messages=messages
        )
        process_flag = True
        while process_flag:
            message = response.choices[0].message
            finish_reason = response.choices[0].finish_reason

            if finish_reason == 'stop' or not message.tool_calls:
                print(message.content)
                process_flag = False

            elif finish_reason == 'tool_calls':
                messages.append(message)
                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)
                    tool_call_id = tool_call.id

                    print(f"Calling tool {tool_name} with args {tool_args}")

                    # Tool invocation through the MCP client session
                    result = await self.session.call_tool(tool_name, arguments=tool_args)
                    result_text = "".join(
                        block.text for block in result.content
                        if hasattr(block, 'text')
                    )
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call_id,
                        "content": result_text
                    })

                response = self.openai_client.chat.completions.create(
                    model='gpt-5.1',
                    tools=self.available_tools,
                    messages=messages
                )
                if response.choices[0].finish_reason == 'stop':
                    print(response.choices[0].message.content)
                    process_flag = False

    async def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        while True:
            try:
                query = input("\nQuery: ").strip()
                if query.lower() == 'quit':
                    break
                await self.process_query(query)
                print("\n")
            except Exception as e:
                print(f"\nError: {str(e)}")

    async def connect_to_server_and_run(self):
        server_params = StdioServerParameters(
            command="python",
            args=["research_server.py"],
            env=None,
        )
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                self.session = session
                await session.initialize()

                response = await session.list_tools()
                tools = response.tools
                print("\nConnected to server with tools:", [tool.name for tool in tools])

                # Convert MCP tool schema to OpenAI function-calling format
                self.available_tools = [{
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                } for tool in tools]

                await self.chat_loop()


async def main():
    chatbot = MCP_ChatBot()
    await chatbot.connect_to_server_and_run()


if __name__ == "__main__":
    asyncio.run(main())


Overwriting mcp_project/mcp_chatbot.py


## Running the MCP Chatbot

**Terminal Instructions**

- به پوشه `mcp_project` برو: `cd mcp_project`
- پکیج‌های مورد نیاز را نصب کن (اگه قبلاً نصب نکردی): `pip install openai python-dotenv nest_asyncio mcp arxiv`
- چت‌بات را اجرا کن: `python mcp_chatbot.py`
- برای خروج از چت‌بات، `quit` بنویس.
- اگه query زدی و می‌خوای پوشه `papers` رو ببینی:
  1) روی `File` کلیک کن
  2) روی `Open` کلیک کن
  3) پوشه `mcp_project` را باز کن کن.

## Resources

- [Quick Start for Client Developpers](https://modelcontextprotocol.io/quickstart/client)
- [Writing MCP client](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)
- [Another mcp chatbot example](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)